<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Turn-taking and interruptions

**[Turn-taking and interruptions](https://learning.nextia-ai.com/courses/speech/m08/turn-taking-and-interruptions/)** · Speech AI: Transcribe, Understand and Speak · Module 8, lesson 3 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the pauses inside a caller's turn with a voice activity detector, see when an end-of-turn wait cuts a caller off, measure the trade-off on every recorded caller turn, and work out what a caller heard before they interrupted the voice.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. The voice detector is Silero VAD, which comes with faster-whisper. |
| **You should know** | The latency budget of a voice agent, from [The voice-agent pipeline](https://learning.nextia-ai.com/courses/speech/m08/the-voice-agent-pipeline/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m08/turn-taking-and-interruptions/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M08-L03-turn-taking-and-interruptions/turn-taking-and-interruptions-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, faster-whisper 1.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. faster-whisper brings its own engine (CTranslate2) and the Silero voice detector: the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'faster-whisper']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 3 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/ivr/calls.jsonl": "5ee1fbad92c79b73fa908076325f2ce6d4030ed12439dacc26b6dcd9bc00d828",
    "data/recordings/agent/ivr_replay.jsonl": "d4d8dd6de71037e7ab6c2ca0433214ec3cfec5112a2de42283c4933994372438",
    "data/recordings/agent/turns.jsonl": "8872c2bccfd8a5987794321df4f859c3628bc1e4123176b5a128dba884a43e3d",
    "data/ivr/I-001/01.wav": "eb1f42c3352fd6d4c31bc20a0e03661aba6acfcd572d6207241c1ce3599daf9d",
    "data/ivr/I-001/02.wav": "7bb3d2f48289d5333fe885be65d9386c5116ffe84e26c00a47bd0f95d55f7c37",
    "data/ivr/I-001/03.wav": "0e1c7eeff8b57dec83d5fd2211a57c143bae5698b309ca9dd8e61ae9e6d0763f",
    "data/ivr/I-001/04.wav": "5a45c738b1ef0996c0fade52cee4bec18a7a6cdc45577d1bfb306a503b619a35",
    "data/ivr/I-002/01.wav": "5a9802e7bc487212cd6789ad81617df7ce957d9676018906965188b764f07328",
    "data/ivr/I-002/02.wav": "5fdf98206c3ae843d39c1cb06126af2ca9a66dd7081425f588535ab5a8e71819",
    "data/ivr/I-002/03.wav": "ee5c046fb7bf5421c613f0c107f84acc6e2342a1ba76eec08a1c1db5ab7faf5e",
    "data/ivr/I-002/04.wav": "845471c165336e2a38a43a805b1368f7a3019f04c151001d0716bf9e4176ea6a",
    "data/ivr/I-003/01.wav": "f06279ba1d2cfa9bb8b8200237db6382692e540b3f5b16c09c52e2fe596c69e7",
    "data/ivr/I-003/02.wav": "027c891e1e7edee2392f40453095c17d789cd216505de3baedb87e23759e84df",
    "data/ivr/I-003/03.wav": "70b31b1b77ce634b7b316d1e6c0c8a347ac3c5a2b416f3de22b522dc343dba9e",
    "data/ivr/I-003/04.wav": "7043a1628796cacd28d4c07ef6aabb0775f6646ce96d44b28ab9486070deef66",
    "data/ivr/I-004/01.wav": "22371302eacf43b9431faca76dfae8f9ee6f34f89396fc13437dfedd251bb20f",
    "data/ivr/I-004/02.wav": "12b83a61e138789d82467bce83e1a4049975f27e417b5dfff5ced3abd162e6ad",
    "data/ivr/I-004/03.wav": "d9642017593e2b1a52afd1ce7a78acd79d65ff4fb00bae9636e92327ad6d96d1",
    "data/ivr/I-004/04.wav": "fb01ecf109ef4e6dc0be17eba8490e089e0e5bf19e65f5ae5d325014576b0b8c",
    "data/ivr/I-005/02.wav": "e8070d87916cd8efd9927b4322d3efe772c349e9e132ec2daf40bd4e42add611",
    "data/ivr/I-005/03.wav": "90fe8c6b0fc0deb68444eb18000af5d59040862079f9699fdbc878c0e6e7ccf9",
    "data/ivr/I-005/04.wav": "f23b3fd84f46467ca8a85aecadb5b88cee563f9685ac7fda4daa7830b1086054",
    "data/ivr/I-006/01.wav": "f28cbdea2cea47d70d613124bf05b2d979edf8faa9bde6ab28a01868a12a1965",
    "data/ivr/I-006/02.wav": "e297e216dfc8abe50b3d9e8f31df623077a2e67a2717059500b3575cdcf8db83",
    "data/ivr/I-006/03.wav": "a1e4754c69c8be248c22a7fa50374d4656159a7ceae78d887c2c3b15415609f9",
    "data/ivr/I-006/04.wav": "0ea77d1ec613df4f8eaed6d23971b04c6c1ec9771b1cea3c428c131fa320a739",
    "data/ivr/I-007/02.wav": "c99f79878c7a46c21493cfe1a088b621bff715ba432175803b3917c0182e5111",
    "data/ivr/I-007/03.wav": "6125daa6c3e09a813201e99eccdedaa94318e9f35f434b05a9f25559704ab184",
    "data/ivr/I-007/04.wav": "e5c9a5b04c51774ec4b94aeb5b41bf377da7bc3b0a06f707d58084289f6b299f",
    "data/ivr/I-008/01.wav": "251a006ad0376923d34a43d3799cbd09b6018c4a25ea3d02a419d936958e15ef",
    "data/ivr/I-008/02.wav": "8fa1e7bc52f0419df9d84b1cd9216dfc8ff9ef208bd44f851681ae3770ad22a2",
    "data/ivr/I-008/03.wav": "b8b4e83dcc28e244e6e5ca686bd546d4af0677e778fce2923a79eba3f5c35fe8",
    "data/ivr/I-008/04.wav": "328d53e574e1ae276a3b67690ff4c166a215faa8029cb797b9bdbf1a4faa89a0",
    "data/ivr/I-009/01.wav": "3944ad8bb7725035c91a02140940dbc8c5382eded0c7f1234e5d42b16dfd41c9",
    "data/ivr/I-009/02.wav": "f72b081c661a3b763a7047ae92610be20db25ae562b351aaab17634c2701372c",
    "data/ivr/I-009/03.wav": "cfc46113d3adf545c3d92744d8e8e964cb6f54f195f217d8c17f55542b65e45d",
    "data/ivr/I-009/04.wav": "386373bf0ea7c79aff21cfe238de1d4de729c27d5fd8b61c0962f0438b0e5fde",
    "data/ivr/I-010/01.wav": "e053484da578bc3dfb40b18f7f2078e3224ab1f2c8a18549cd62a8f97c0563f8",
    "data/ivr/I-010/02.wav": "ff0255c8dce8691bf5f0860e8ae7b8077bc0f586ebf2e7e1dfb1d5bbba5cd51b",
    "data/ivr/I-010/03.wav": "d9e0aef67dd8ba7066d5995886eab5b89058bfc50c3e37b61d1cad3a3c088db3",
    "data/ivr/I-010/04.wav": "4d9eb817f74fab94f521235939a368e879f001b3886d72ff6775a31b9c8f5b5e",
    "data/ivr/I-011/01.wav": "8c6b90067a7c6d0ae4f8d1d6655469640e37fa1559a7d542614a1974d0031c44",
    "data/ivr/I-011/02.wav": "1212f396dcb0cf2d6d71bf1265c4871dabbc93c7aea504a5f518c23349789723",
    "data/ivr/I-011/03.wav": "4143f4a423b48ebb16b7e296bc8c5facf87ced9312da3e25f1ad5244bb2cbc63",
    "data/ivr/I-011/04.wav": "91c7ce24d4442678f8d3d45457c482b1f9b5b49a1138d6fd3047e9c6170695da",
    "data/ivr/I-012/01.wav": "9dcb2a9ec04a7bb2000085d54fd500d9b9799886c968594133ab90a8c584f766",
    "data/ivr/I-012/02.wav": "55fd250c110bd8a146c070efe963e662e21c8374d97fb0e7c1712360520f6c45",
    "data/ivr/I-012/03.wav": "ac9636976da6b6a11680e5f1cc8ff2f38c76f254a9a7d5d3beb26e1f8c84fcbe",
    "data/ivr/I-012/04.wav": "95615c6f014bb1a2e4fc22ad4ee6e705dd7c3e03a88cc4aaee8e810fddbb91d9",
    "data/ivr/I-018/01.wav": "5ca0249c39d5e78759a3fdcc4a64f421ad1829bfba2ded2dafde3bde763f4d56",
    "data/ivr/I-020/01.wav": "d1d3eb1056f68aae4c8e1d2bf7d40388cb360d8b4742e58e5ba6cfc339d20eb2",
    "data/ivr/I-025/02.wav": "0152453019e46ba8895429a5eca17ec40611adf766b2e99f85853e9a54ab44d7",
    "data/ivr/I-026/02.wav": "9536383ad402522c75485d87a1cf4a3e9a62e152d456a3291ad53dbd212df368",
    "data/ivr/I-027/02.wav": "d286717521bb3de9e57dd7464b1af0b308adbf6dd384a51fcc940c0035fc40f7",
    "data/ivr/I-028/02.wav": "85e661d1e9a6d13d03f49bec5d599ed7d8a581548ff0805328478ac61a67f763",
    "data/ivr/I-029/01.wav": "f77c32895ddfbde41ac7f166cff3886c1c0397d876411c20b3495b2ac87a7b4a",
    "data/ivr/I-029/02.wav": "d5253bde00f3cccd36b73ae45c12f5eddacb0b45206bad999b721132b166285b",
    "data/ivr/I-030/02.wav": "7d234a840cd254d4de39824ac1fe753a50a011f2889400b9fdacf206740b513a",
    "data/ivr/I-034/01.wav": "3afd0aeb8f071b19f736fcb4b18aec892da1ac799ff499f3add40586a5a3b231",
    "data/ivr/I-035/01.wav": "3a8c274792c16f4afb7b7ac0f1186f0daa50d21fbe9aaab87e29f93b66de36c9",
    "data/ivr/I-037/01.wav": "c8f1c120cdc4b835253e1f3a8487f7472b7fe12b3cffdc1173da047561e9b580",
    "data/ivr/I-037/02.wav": "b0c8ee76476ccf6671be1e73009d691e0b500b2d3538d93e1642fb50cd77811e",
    "data/ivr/I-037/03.wav": "347422784c41fcae58126d838543aea4dbc5621ecf49c0ae1ce533b3b8c1e28b",
    "data/ivr/I-037/04.wav": "181dd81ccff1e49f78063a47c2c3ffbbf46366467d105196f81eb868c1128d6c",
    "data/ivr/I-038/01.wav": "131cd2077eb21941735d1ff67a39131873da8b3a08ca096e0b97c1a98ae1d66d",
    "data/ivr/I-038/02.wav": "d09658aa605cb417fa1bb38cb8f3c342cd1e214211dd108170b2de8fd4bb5c42",
    "data/ivr/I-038/03.wav": "5d55a495d0ebc3a64483023c896d2301d091968995039490c30aae529855a40c",
    "data/ivr/I-038/04.wav": "bde860baf6d95cb17387030339d2fa380d80f60cd6f5b656372b0b9feff5ac48",
    "data/ivr/I-039/02.wav": "6cf823a8d673242042a4c2c99dbffcd9d271fd75049469c69916d6d4b76e5da1",
    "data/ivr/I-039/03.wav": "9462ee31326d6f15c720f73142ef8130d4835884cd01c530c830784aec727071",
    "data/ivr/I-039/04.wav": "738540fcc5c72b33219bba85b756b1388812af222de89e822f7e798f3d2778bf",
    "data/ivr/I-040/01.wav": "f0662f5a7405f9eb12e541792af10da52e86c83c3ef5443d8fd7f8a56ec36121",
    "data/ivr/I-040/02.wav": "7d5385c97f7ad82f7eeda8afea0d1047161451b986231f5781c0c6588ec1f119",
    "data/ivr/I-040/03.wav": "80a3f79e6d43898b3be0e27fbd44183d33cb3bad86ce802e4c7d27905cdeb630",
    "data/ivr/I-040/04.wav": "33d0d83d7de6147bf77e0242f41cc35510730281d0a03a98e6f71db3ca4c4dc0",
    "data/ivr/I-041/01.wav": "20d3fcf13553a8b22dc218b4968135d608c8d8b8fedaa8795697601ff27d5b57",
    "data/ivr/I-041/02.wav": "b90173e6dc401bca7d72b61e9cef81ba8a216601ca169ea3b1b6307440eac06e",
    "data/ivr/I-041/03.wav": "5d9a1272ee71000a2cdfcb69c06df92429e6f16a485655780e518dfe3c085cd3",
    "data/ivr/I-041/04.wav": "c9ce6f3148d535841b21c0fb84e3edecfe6de68f0721543bb107fbfa6ab6b356",
    "data/ivr/I-042/01.wav": "ce7d0ee7d7caa2701b3b2e79a5d12d18634a854ee9949e20c8ecf2ac36cff596",
    "data/ivr/I-042/02.wav": "b87c9180f53426fd3eae9681ebb0c3131b51a90de9808204765ad095cb951a20",
    "data/ivr/I-042/03.wav": "c21dcf45db0172b57b54e3ec0c10956106b033e0f80e55913151cfb53b9c280e",
    "data/ivr/I-042/04.wav": "707277a0092c0763ea5929bc7465b098c8de6516cb5f946aa8f9c8fe85895d25",
    "data/ivr/I-045/02.wav": "79451d2dbd53227eec16a3151923a5eb6eec11d587763492f0f8043f43fa22d4",
    "data/ivr/I-045/03.wav": "308526d87b223ec3cc95122259d6ec7e3a8fc314912e1d3fedba9914e55c2fcf",
    "data/ivr/I-045/04.wav": "6983156116714a809e1da67d49c719980ad1c3639c595ee1b41d650e7a1412fc",
    "data/ivr/I-046/02.wav": "089a7887f8f21526bfaf8ceb598caff30ec52d78ddcff82537f9203b4ca2867e",
    "data/ivr/I-046/03.wav": "b37909be65b38da3b7833fc61ba4b2fbe35e12a156c30335323bd0de2a445844",
    "data/ivr/I-046/04.wav": "93a3f99435476e59d91b6ad612c2e038f035003ff7489efbfbf31fabdd9fff8d",
    "data/ivr/I-047/01.wav": "b573100f06b59d3c6b24afd1fa4baabfbbe93fa79e701cd2c7454f4aae1b5d83",
    "data/ivr/I-047/02.wav": "ef50ef5b4937c71c2fbb03e3d0317953cc605c21bed2f6296754d08f22db7c21",
    "data/ivr/I-047/03.wav": "88ae5a9f7df3ca1a33a3e690815843d24fceb4eff10cedd3a423405591a8d1aa",
    "data/ivr/I-047/04.wav": "53b886e857bae84c768cfb751f1fab4c735e01f9637dc5a05b13c174980f8fd7",
    "data/ivr/I-049/02.wav": "af3c48360ff85d1776b6cab1c7c4c4c6898cd1cf2e36d773e06281461b11bcb1",
    "data/ivr/I-049/03.wav": "14e6c9aed8e246755578b9bf2d5c51ce9dc975592b6c49b62a1ab29b98952a15",
    "data/ivr/I-049/04.wav": "de9d60f481c9cac2805dc77b2c936c3d76b1c76ab5043e3d9ae3622454233f87",
    "data/ivr/I-050/01.wav": "a7133c7e8381850dc52770c228e5e25651ac92fdb22132a4e02ff5fcfc069b71",
    "data/ivr/I-050/02.wav": "5e696c83079982f2afd27108c7f3109e5e7c491f3e9edc6a565a3e0bbc3a28c0",
    "data/ivr/I-050/03.wav": "7a4256fd0199b76189b147e652582401e727d9bf1434e02f5852cefd6d74bb83",
    "data/ivr/I-050/04.wav": "050cfe2680d59aaf37ac9c9211c79c28ec81632a465a90bfeef6269b1d9fe52a",
    "data/ivr/I-051/02.wav": "d2925a82d02528f3f6d33787cf49cf36bff204da555184c95608cfc7fbdfa2d3",
    "data/ivr/I-051/03.wav": "bd02d9793d3d6167481fec6039af452c907f99af7ca4e358a729aea9c319a434",
    "data/ivr/I-051/04.wav": "55b74f99598cddaf99ee8717a7cca47141b18697c8dfd7c6c8af62a32da5a4a2",
    "data/ivr/I-052/02.wav": "fd7f1ed843cd44241c43801a1cfa20b73c996aae5bd32fcf66fa5dec60c554db",
    "data/ivr/I-052/03.wav": "9f4715cdbe3857015bce71ff60b8ccaefec71a21455f837ae39b20c5ed0efe5b",
    "data/ivr/I-052/04.wav": "20fce9b0b13ede24d86256e0dc22f313567a91e4a957a56237de7231497c382b",
    "data/ivr/I-054/02.wav": "fc3a9b23a5ec09bd5039fbebce7b1529e1c9fd070bd7d84b285739e86696cf82",
    "data/ivr/I-054/03.wav": "e328ac8e36dc0a490a86e87d5b5a061f554cea86cbff529f5220343762c04c59",
    "data/ivr/I-054/04.wav": "c3cfde9ca26570eb4464227d6a8bbe70c20175cdd2b7856e591bf70ad03f7d36",
    "data/ivr/I-054/05.wav": "fdac42edf138c674410dd9ec3edc128895c6f36ef24f0ba8982432a68fc84e82",
    "data/ivr/I-055/02.wav": "4a6983250bcc63fe55dbdcd08a2add0551bb53652f4b758acb337301e90e9dae",
    "data/ivr/I-055/03.wav": "806112fedabde1cf99244da0506d3fd219041488608f3512f6ec3d01cd1448fc",
    "data/ivr/I-055/04.wav": "34e2350916a3dbf3304ad8ccc73229386e7a9797705e95b94b56a868cade737d",
    "data/ivr/I-055/05.wav": "08453213b771a2a30e1370b33ff81b384b41599fbeae9d35724fa84767474c3d",
    "data/ivr/I-056/02.wav": "8f5714baa6e63a67851bf94e954556b95208d968a417a1f58244f5190221e31d",
    "data/ivr/I-056/03.wav": "3efc2b2edfa2c9c3c6253e4caf0560c548eb3c1d77fc2c405a2bdff5c7f8f200",
    "data/ivr/I-056/04.wav": "e2c72d9eac862fdc233f8b853a5659855884e85898ecc923cf4e1be6fb1e55b6",
    "data/ivr/I-056/05.wav": "bf68e7ed14b6a5a5e518a9ec827aa67972439e6ae6911fc30c8f01ead2d80843",
    "data/ivr/I-058/01.wav": "d9a619b4398ca6a8b35b036d2cadece893b4d9bd4694bdbf791db3720ec56ac6",
    "data/ivr/I-059/01.wav": "518619e0e4b894a438921aaf5652180c4a50431e6f7717eda8bfcd31e411580d",
    "data/ivr/I-060/01.wav": "7b1f0070f84d46418d275b330637a6e3d17e3f5608710729db321ce8e37404e6",
    "data/ivr/I-064/01.wav": "5addce776e5547d23f91d0a3820b47616a87d8217eed0e4ebdce80a7552988d5",
    "data/ivr/I-066/01.wav": "27667a941ca04092b03901d3a3b7833bf602092092f822a0dd7af1a93370ae5b",
    "data/ivr/I-066/02.wav": "28cfa426231ecb8430ca547af784540c4c563a685bbee54d666de3ebb13bb9ca",
    "data/ivr/I-067/01.wav": "4146a3eea0b2dec86137923165eb3006da429840b0079ca15ab3246395eee9f5",
    "data/ivr/I-067/02.wav": "4812c45559c0a7b9d578067a78c2d91e6687ebd0f47cc0730c0d961893980cf5",
    "data/ivr/I-068/01.wav": "33022c860b4bf1983174ac3e425c69389e262e99fbb8a9f56472fcaf5dca5105",
    "data/ivr/I-068/02.wav": "78c1c218335730ad89856a02f284cf2aefd70947f85f14a99ed545c8723d230d",
    "data/ivr/I-069/02.wav": "d4b633601df08778d57a1cf8042232efc5e045188ae72395775413a7ce0b2da2",
    "data/ivr/I-072/02.wav": "a4af1f0f92fb126299b132dfbb2039fded95861afe193bf6763edc360409ebb2",
    "data/ivr/I-073/01.wav": "20a222cc8e6af1fddb4ff19b6dccdd1831fa41763bb6e854c6e47f738f022693",
    "data/ivr/I-073/02.wav": "2cedd12d36aeb573d390c82140ccca6eb427c6554acf7935c8c20252909eb651",
    "data/ivr/I-074/02.wav": "02cc83a6f7fba1679b5f2f5d1e56b29db233c715741bce97f11cdf1b05b3d6e3",
    "data/ivr/I-076/01.wav": "fe873db34b17913324edf9299d081a28bda6d818fce46bfe643814f80533ebbf",
    "data/ivr/I-076/02.wav": "5e2d2fdaf496b48a613667fa6f51b105c35743d860e55e1313f656b55841f361",
    "data/ivr/I-076/03.wav": "32f6f5399edd3e43535e224a1c6aaac7108afa44c673e53b721835ca76bcd822",
    "data/ivr/I-077/01.wav": "50f999c2e22defdaf229842b6ca87ffe2bf053daac630a0dce344575013ab726",
    "data/ivr/I-077/02.wav": "e0a3ab25f0d58cb3d158ed68374f4e65ef69589b0efc5e1849bbefb53a630ea4",
    "data/ivr/I-077/03.wav": "b0c283ed5161558df684a1125f8792f7441f012d347de24634cc70f646177fce",
    "data/ivr/I-078/02.wav": "552ee564eb702ec249907e1c84e90e3e4e78ba2fe3d29534fb7ce8fd9d000193",
    "data/ivr/I-078/03.wav": "617c4443eb1bf2f27773542c7ecef25b0b96eb6a5f8453a82c0a257c2480f4c0",
    "data/ivr/I-079/01.wav": "2748835c83d2d89ed2a650a9aef03a2c802f638651b93d7400414137118957d4",
    "data/ivr/I-079/02.wav": "c6862bcf185a88be56aa31b8e513b0e10edfc84e7acc3eecdf7528ad8f6bfa26",
    "data/ivr/I-079/03.wav": "19d8f789079b34f0d8b2903ea36690795ceec7975b297e89651c82b84320d718",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A caller who pauses

A voice agent must decide when the caller has finished. It waits for a silence of a fixed length: the **end-of-turn wait**. A short wait answers sooner. But some callers pause in the middle of a sentence, for example to read an order number from a paper.

The IVR test calls are recorded caller turns. Run the next cell. It loads them and picks one turn of call I-049.

In [ ]:
ivr_calls = [json.loads(line) for line in open("data/ivr/calls.jsonl", encoding="utf-8") if line.strip()]
spoken = [(c["call_id"], n, c["kind"], i) for c in ivr_calls for n, i in enumerate(c["inputs"]) if "say" in i]
print(len(ivr_calls), "IVR test calls;", len(spoken), "spoken caller turns (the others are keypad tones or silence)")
call_id, n, kind, turn = next(s for s in spoken if s[0] == "I-049" and s[1] == 2)
print(call_id, kind, "| the caller says:", turn["spoken"], "| the order number:", turn["say"])

> **Check.** 60 IVR test calls and 141 spoken caller turns. In call I-049, the caller reads the order number in two parts, with a pause.

## 2. Find the pauses with a voice detector

The next cell runs Silero VAD on the turn, with the same settings as the IVR's engine. It prints the speech parts and the pause between them, and plays the turn.

> **Predict.** How long is the pause between "seven nine eight" and "zero nine zero"?

In [ ]:
from faster_whisper.vad import VadOptions, get_speech_timestamps

# The IVR engine's own settings: a speech probability over 0.5, silences from 100 ms, no padding.
OPTS = VadOptions(threshold=0.5, min_silence_duration_ms=100, speech_pad_ms=0, min_speech_duration_ms=150)

def speech_parts(path):
    """The (start, end) seconds of each speech part of a caller's turn, and the turn's length."""
    x16, _ = read_audio(path, sr=16000)
    return [(t["start"] / 16000, t["end"] / 16000) for t in get_speech_timestamps(x16, OPTS)], len(x16) / 16000

parts, seconds = speech_parts("data/ivr/" + turn["file"])
pauses = [b[0] - a[1] for a, b in zip(parts, parts[1:])]
print("speech parts:", [(round(a, 2), round(b, 2)) for a, b in parts], " turn length", round(seconds, 2), "s")
print("pauses inside the turn:", [round(p, 2) for p in pauses])
display(Audio("data/ivr/" + turn["file"]))

> **Check.** Two speech parts, and a pause of 1.95 s between them. Listen: the caller stops, then goes on.

## 3. Where does each wait end the turn?

The rule of the pipeline: the turn ends after the first silence that is at least as long as the wait. Run the next cell for four waits.

In [ ]:
def turn_end(parts, wait):
    """When the pipeline decides that the turn is over: after the first silence of `wait` seconds.
    Returns (the time, cut) where cut is True when speech was still to come."""
    for (a, b), (c, d) in zip(parts, parts[1:]):
        if c - b >= wait:                     # a pause at least as long as the wait: the turn "ends" here
            return b + wait, True
    return parts[-1][1] + wait, False

for wait in (0.3, 0.8, 1.5, 2.0):
    end, cut = turn_end(parts, wait)
    print(f"wait {wait} s: the turn ends at {end:.2f} s  ->  {'cut in the pause' if cut else 'after the whole number'}")

> **Check.** With a wait of 0.3 s, the turn ends at 2.19 s, in the pause: the agent would answer "seven nine eight" only. With 0.8 s (the IVR's setting), it ends at 2.69 s, still in the pause. Only a wait of 2.0 s keeps the whole number, and then the turn ends at 6.77 s: every caller waits 2 seconds before each answer.

## 4. Every caller turn: the trade-off

One call is an example. The next cell finds the longest pause inside each of the 141 spoken caller turns (it takes a few seconds), and counts the turns that each wait would cut.

In [ ]:
rows = []
for call_id, n, kind, turn in spoken:
    p, _ = speech_parts("data/ivr/" + turn["file"])
    longest = max([b[0] - a[1] for a, b in zip(p, p[1:])], default=0.0)
    rows.append({"call_id": call_id, "kind": kind, "number": turn["say"].isdigit(), "longest_pause": longest})

waits = [round(0.1 * k, 1) for k in range(2, 16)]
numbers = [r for r in rows if r["number"]]
cut_all = {w: sum(r["longest_pause"] >= w for r in rows) for w in waits}
cut_num = {w: sum(r["longest_pause"] >= w for r in numbers) for w in waits}
for w in waits:
    print(f"wait {w:.1f} s: cuts {cut_all[w]:3d} of {len(rows)} turns ({cut_all[w] / len(rows):4.0%});  "
          f"order numbers {cut_num[w]:2d} of {len(numbers)}")

> **Check.** A wait of 0.3 s cuts 13 turns, and 9 of the 31 order-number turns. A wait of 0.8 s cuts 4 turns, and 2 order numbers. Even a wait of 1.5 s still cuts 2 turns: some callers pause for longer. No fixed wait is right for every caller.

The next cell draws the two curves.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(waits, [100 * cut_num[w] / len(numbers) for w in waits], marker="o", label=f"order-number turns ({len(numbers)})")
ax.plot(waits, [100 * cut_all[w] / len(rows) for w in waits], marker="o", label=f"all spoken turns ({len(rows)})")
ax.set_xlabel("end-of-turn wait (seconds of silence)"); ax.set_ylabel("turns cut early (%)")
ax.legend(); ax.set_title("A shorter wait answers sooner, and cuts more callers off"); plt.show()

## 5. Failure case: the turn ends in the middle of a number

**Symptom:** with the IVR's wait of 0.8 s, call I-049's turn ends after "seven nine eight". A pipeline that does not know better looks up order 798, and finds nothing.

**Fix:** use what you know about the answer. An order number has six digits. The IVR checks it: when it hears only three, it says "Please say the rest of the number". Run the next cell: it prints the IVR's recorded call.

In [ ]:
replay = {r["call_id"]: r for r in (json.loads(line) for line in open("data/recordings/agent/ivr_replay.jsonl", encoding="utf-8") if line.strip())}
for t, what, detail in replay["I-049"]["timeline"]:
    if what in ("heard", "say", "end"):
        print(f"{t:6.2f} s  {what:5}  {detail[:95]}")

> **Check.** The IVR heard "Seven nine eight.", asked for the rest, heard "090.", read the whole number back, and gave the order status. A check on the **content** (is the number complete?) fixes what a fixed wait cannot. A voice agent needs the same rule, in code.

## 6. When the caller talks over the voice

A **barge-in** is a caller who starts to talk while the voice is still speaking. The pipeline should stop the voice at once, and remember which words the caller really heard. The next cell takes one recorded agent answer (call I-054) and its length in audio.

The moments when the caller "talks" are constructed: the replay's callers never interrupt. The cell estimates the words heard at each moment.

In [ ]:
turns = [json.loads(line) for line in open("data/recordings/agent/turns.jsonl", encoding="utf-8") if line.strip()]
answer = next(t for t in turns if t["call_id"] == "I-054" and t["input"] == 2)
print(repr(answer["reply"]), "-", answer["reply_audio_s"], "s of audio")

def played_text(reply, audio_seconds, stop_at):
    """The words that the caller heard before the voice stopped at `stop_at` seconds. An approximation: the voice is
    taken to speak the characters at an even speed (a real voice gives word times; Piper's can)."""
    share = min(1.0, stop_at / audio_seconds)
    cut = int(len(reply) * share)
    return reply[:cut].rsplit(" ", 1)[0] if share < 1 else reply

for stop_at in (1.0, 2.0, 3.0):
    print(f"the caller talks at {stop_at} s -> heard: {played_text(answer['reply'], answer['reply_audio_s'], stop_at)!r}")

> **Check.** The answer lasts 5.0 s. A caller who talks at 2.0 s heard only "Your order number is six". If the agent keeps the whole answer in the call's history, it believes that the caller heard the question "Is that correct?", and may take the next words as the answer to it.

## 7. Your turn: the shortest safe wait

> **Your turn.** Larkfield accepts that at most 3% of the spoken caller turns are cut early. Find the shortest wait in `waits` that cuts at most 3% of all spoken turns (use `cut_all` and `rows`). Put it in `shortest_wait`.

In [ ]:
shortest_wait = min(w for w in waits if cut_all[w] / len(rows) <= 0.03)
print(shortest_wait, cut_all[shortest_wait], "of", len(rows))

In [ ]:
expect("shortest_wait", shortest_wait, 0.7, tolerance=1e-9)

> **Check.** The shortest wait is 0.7 s: it cuts 4 of the 141 spoken turns. That is 0.1 s shorter than the IVR's 0.8 s, so the latency budget gains 0.1 s. A wait of 0.3 s would gain 0.5 s, but cut about three times more turns. Remember that every call here is synthetic: real callers pause in other ways. Measure the pauses on your own calls before you choose.

## Next

You found the pauses inside callers' turns, measured what each end-of-turn wait cuts, and saw how a content check and a barge-in rule help. The [lesson page](https://learning.nextia-ai.com/courses/speech/m08/turn-taking-and-interruptions/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Speech-to-speech models and evaluation](https://learning.nextia-ai.com/courses/speech/m08/speech-to-speech-models-and-evaluation/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m08/turn-taking-and-interruptions/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.